In [21]:
import pandas as pd
from modules import Agent,Model
from DCAStrategy import DCAAgent
from LSStrategy import LSSAgent
import numpy as np
import torch
import pickle
device = "cuda" if torch.cuda.is_available() else "cpu"
import matplotlib.pyplot as plt
import seaborn as sns
import os
from preprocessing import Feature_Extractor, data_preprocessing


Start Trading

In [28]:
symbol = "MBB"
from_date ="2024-12-25"
to_date = "2025-01-10"
# from_date ="2023-05-18"
# to_date = "2024-06-18"
# from_date ="2023-01-15"
# to_date = "2023-12-01"
skip =1

In [29]:
df = pd.read_csv(f'DataTrading/{symbol}.csv')

In [30]:
df.tail

<bound method NDFrame.tail of            Date    Close     Open     High      Low
0    2023-12-25  15655.0  15613.0  15783.0  15570.0
1    2023-12-26  15698.0  15655.0  15740.0  15570.0
2    2023-12-27  15655.0  15740.0  15740.0  15613.0
3    2023-12-28  15783.0  15655.0  15825.0  15613.0
4    2023-12-29  15868.0  15825.0  16038.0  15783.0
..          ...      ...      ...      ...      ...
257  2025-01-06  21217.4  21478.3  21652.2  21217.4
258  2025-01-07  21650.0  21700.0  21700.0  21200.0
259  2025-01-08  21600.0  21450.0  21700.0  21450.0
260  2025-01-09  21500.0  21600.0  21700.0  21450.0
261  2025-01-10  21150.0  21450.0  21500.0  21150.0

[262 rows x 5 columns]>

Trade without LSTM

With LSTM

DCA without LSTM

In [31]:
df = pd.read_csv(f'DataTraining/{symbol}.csv')
df['Date'] = pd.to_datetime(df['Date'])
df_init = df[['Close']]  
df_init = data_preprocessing(df_init, Feature_Extractor)
real_trend = df_init['Close'].tolist()
parameters = [df_init[cl].tolist() for cl in df_init.columns]
# initial_money = np.max(parameters[0]) * 5
initial_money =  1000000
minmax = pickle.load(open(f"checkpoint/{symbol}_DCAscaler.pkl", 'rb'))
scaled_parameters = minmax.transform(np.array(parameters).T).T.tolist()
with open(f"checkpoint/{symbol}_DCAmodel.pkl", 'rb') as fopen:
    model = pickle.load(fopen)
df = df[['Date', 'Close']]
#Preprocess Dataframe
df = data_preprocessing(df, Feature_Extractor)
selected_data = df.loc[(df['Date'] >= from_date) & (df['Date'] <= to_date),:]
data_list = selected_data.values.tolist()

agent3 = DCAAgent(model = model,
                timeseries = scaled_parameters,
                skip = skip,
                initial_money = initial_money,
                real_trend = real_trend,
                minmax = minmax,
                window_size = 10)

trade_results = []
for row in data_list:
    #ensure first column is Date
    date = row[0]
    value = row[1:]
    result = agent3.trade(value, date = date)
    trade_results.append(result)

result = pd.DataFrame(trade_results)

# Convert the 'date' column to datetime format
result['date'] = pd.to_datetime(result['date'])
result = result[result['status'] != 'data not enough to trade']
df_action_2 = result[result['action'] == 2]

# Sắp xếp DataFrame theo timestamp hoặc date để đảm bảo lấy hàng cuối cùng
df_action_2_sorted = df_action_2.sort_values(by='date')

# Lấy giá trị total_investment của hàng cuối cùng
last_total_investment = df_action_2_sorted.iloc[-1]['total_investment']
df_sorted = result.sort_values(by='date')

# Lấy giá trị total của hàng cuối cùng
last_total = df_sorted.iloc[-1]['total']
investmentvalue = (last_total-initial_money)/initial_money *100
total_gain = result['gain'].sum()
investGain = total_gain/initial_money *100

print(last_total_investment)
print(investmentvalue)
print(total_gain)
print(investGain)

# Mark buy and sell actions
buy_signals = result[result['action'] == 1]
sell_signals = result[result['action'] == 2]
plt.plot(result['date'], result['close'], linestyle='-', label='Close Price')

plt.scatter(buy_signals['date'], buy_signals['close'], marker='^', color='g', s=100, label='Buy Signal')
plt.scatter(sell_signals['date'], sell_signals['close'], marker='v', color='r', s=100, label='Sell Signal')

# Customize the plot
plt.title(f'{symbol} Gain {round(total_gain,2)} Total Investment {round(investGain,2)}%')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)

# Show the plot
plt.show()

KeyError: 'date'